# EcoShield AI — Streamlit Prototipi

Bu notebook Görev 8 Streamlit uygulamasının ön kontrolünü ve başlatma komutunu sağlar.

Uygulama:

- Görev 6'da seçilen `cat_d8_balanced` modelini kullanır.
- Validation'da sabitlenen threshold değerini değiştirmez.
- Tek işlem JSON girdisi ve toplu CSV tahmini destekler.
- Transaction ve identity CSV dosyalarını `TransactionID` üzerinden left join edebilir.
- Büyük tahminlerde chunk, ilerleme, işlenen satır, süre ve ETA gösterir.
- Model performansını Görev 7 final test çıktılarından gösterir.

Model eğitimi, threshold tuning veya test değerlendirmesi bu görevde tekrar yapılmaz.


## 1. Dosya ve paket kontrolü

In [4]:
import importlib.util
import subprocess
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "streamlit_app.py").exists():
    candidate = NOTEBOOK_DIR / "notebooks"
    if (candidate / "streamlit_app.py").exists():
        NOTEBOOK_DIR = candidate

PROJECT_ROOT = NOTEBOOK_DIR.parent
APP_PATH = NOTEBOOK_DIR / "streamlit_app.py"
MODEL_PATH = PROJECT_ROOT / "models" / "heavy" / "optimized_single_heavy_model.joblib"
SELECTION_PATH = PROJECT_ROOT / "outputs" / "metrics" / "selected_single_heavy_model.csv"
MANIFEST_PATH = PROJECT_ROOT / "outputs" / "metadata" / "common_cache_manifest.json"
FINAL_METRICS_PATH = PROJECT_ROOT / "outputs" / "metrics" / "final_test_comparison.csv"

required_packages = ["streamlit", "catboost", "joblib", "numpy", "pandas", "pyarrow"]
missing_packages = [
    package
    for package in required_packages
    if importlib.util.find_spec(package) is None
]
if missing_packages:
    raise ModuleNotFoundError("Eksik paketler: " + ", ".join(missing_packages))

required_files = [APP_PATH, MODEL_PATH, SELECTION_PATH, MANIFEST_PATH, FINAL_METRICS_PATH]
missing_files = [path for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        "Eksik Görev 8 dosyaları:\n" + "\n".join(str(path) for path in missing_files)
    )

subprocess.run([sys.executable, "-m", "py_compile", str(APP_PATH)], check=True)
print("Streamlit uygulaması Python sözdizimi: OK")
print("Final model:", MODEL_PATH.relative_to(PROJECT_ROOT))
print("Uygulama:", APP_PATH.relative_to(PROJECT_ROOT))


Streamlit uygulaması Python sözdizimi: OK
Final model: models\heavy\optimized_single_heavy_model.joblib
Uygulama: notebooks\streamlit_app.py


## 2. Başlatma komutu

In [5]:
launch_command = f'"{sys.executable}" -m streamlit run "{APP_PATH}"'
print("VS Code terminalinde çalıştırın:")
print(launch_command)
print("\nStreamlit varsayılan olarak http://localhost:8501 adresini açacaktır.")


VS Code terminalinde çalıştırın:
"c:\Users\pc\anaconda3\envs\torchcuda\python.exe" -m streamlit run "c:\Users\pc\Desktop\YZTA-Bootcamp-2026\notebooks\streamlit_app.py"

Streamlit varsayılan olarak http://localhost:8501 adresini açacaktır.


## 3. Kullanım

Terminal komutunu çalıştırdıktan sonra:

1. **Tek işlem** sekmesinde ortak feature şemasındaki alanları JSON olarak girin.
2. **Toplu CSV** sekmesinde birleştirilmiş CSV veya ayrı transaction/identity CSV dosyalarını yükleyin.
3. Tahminleri başlatın ve sonuç CSV'sini indirin.
4. **Model performansı** sekmesinde final test metriklerini ve görsellerini inceleyin.

`isFraud` kolonu yüklenirse tahmin girdisinden çıkarılır. Uygulama hedef kolonunu model girdisi olarak kullanmaz.


# Görev 8 tamamlanma koşulları

- [x] Final D8 Balanced model dosyası yüklenir.
- [x] Validation'da seçilen threshold sabit kullanılır.
- [x] Tek işlem ve toplu CSV tahmini desteklenir.
- [x] Transaction–identity left join protokolü korunur.
- [x] Feature sırası ortak şemaya göre doğrulanır.
- [x] Kategorik eksikler `__MISSING__`, sayısal eksikler `NaN` olarak hazırlanır.
- [x] Uzun tahminlerde ilerleme, satır sayısı, süre ve ETA gösterilir.
- [x] Tahmin sonuçları CSV olarak indirilebilir.
- [x] Final test metrikleri ve görselleri gösterilir.
- [x] Model eğitimi, threshold tuning ve test değerlendirmesi tekrarlanmaz.


In [1]:
import pandas as pd
from pathlib import Path

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

test_path = project_root / "data" / "processed" / "catboost" / "test.parquet"

test_df = pd.read_parquet(test_path)

sample_json = test_df.iloc[0].where(
    test_df.iloc[0].notna(),
    None
).to_dict()

import json
print(json.dumps(sample_json, ensure_ascii=False, indent=2))

{
  "TransactionDT": 86535,
  "TransactionAmt": 15.0,
  "ProductCD": "H",
  "card1": "2803",
  "card2": "100.0",
  "card3": "150.0",
  "card4": "visa",
  "card5": "226.0",
  "card6": "debit",
  "addr1": "337.0",
  "addr2": "87.0",
  "dist1": null,
  "dist2": null,
  "P_emaildomain": "anonymous.com",
  "R_emaildomain": "__MISSING__",
  "C1": 1.0,
  "C2": 1.0,
  "C3": 0.0,
  "C4": 0.0,
  "C5": 0.0,
  "C6": 1.0,
  "C7": 0.0,
  "C8": 1.0,
  "C9": 0.0,
  "C10": 1.0,
  "C11": 1.0,
  "C12": 0.0,
  "C13": 1.0,
  "C14": 1.0,
  "D1": 0.0,
  "D2": null,
  "D3": null,
  "D4": null,
  "D5": null,
  "D6": null,
  "D7": null,
  "D8": null,
  "D9": null,
  "D10": null,
  "D11": null,
  "D12": null,
  "D13": null,
  "D14": null,
  "D15": null,
  "M1": "__MISSING__",
  "M2": "__MISSING__",
  "M3": "__MISSING__",
  "M4": "__MISSING__",
  "M5": "__MISSING__",
  "M6": "__MISSING__",
  "M7": "__MISSING__",
  "M8": "__MISSING__",
  "M9": "__MISSING__",
  "V1": null,
  "V2": null,
  "V3": null,
  "V4": null,
